# Interactive Stock Prediction Visualization

This notebook demonstrates how to load the `FutureStockPredictorEnhanced` module to forecast stock prices and visualizes the results dynamically using Plotly.

In [1]:
import sys
import os
import pandas as pd
import plotly.graph_objects as go

from predict_future_enhanced import FutureStockPredictorEnhanced

# Set configuration parameters
TICKER = "AAPL"
PREDICT_DAYS = 5
HISTORICAL_DAYS = 60

## 1. Fetch Data and Train Model

We instantiate the predictor, fetch historical data and sentiments, and train the GRU neural network.

In [2]:
print(f"Fetching data and calculating features for {TICKER}...")
predictor = FutureStockPredictorEnhanced(TICKER)
df = predictor.fetch_and_prepare_data()

print(f"Training GRU model with {len(predictor.features)} features...")
model = predictor.train_model()

print(f"Simulating future predictions for {PREDICT_DAYS} days...")
last_price, est_price, direction, predicted_prices, future_dates = predictor.predict_future(model, PREDICT_DAYS)

print(f"\nLast Known Price: ${last_price:.2f}")
print(f"Estimated Price:  ${est_price:.2f}")
print(f"Overall Direction:{direction}")

Fetching data and calculating features for AAPL...


Fetching Yahoo Finance news...
Fetching Finviz news...


Fetching Google News...


Fetching LSEG/Refinitiv Analyst Ratings...
Training GRU model with 11 features...


Simulating future predictions for 5 days...

Last Known Price: $258.63
Estimated Price:  $260.84
Overall Direction:Upward


## 2. Visualize with Plotly

We plot the last 60 days of historical `Open` prices alongside our newly generated predictions.

In [3]:
hist_dates = predictor.df["Date"].iloc[-HISTORICAL_DAYS:].tolist()
hist_prices = predictor.df["Open"].iloc[-HISTORICAL_DAYS:].tolist()

# Connect the timeline so the line is continuous
plot_dates = [hist_dates[-1]] + future_dates
plot_prices = [hist_prices[-1]] + predicted_prices

fig = go.Figure()

# Add Historical Data
fig.add_trace(go.Scatter(
    x=hist_dates,
    y=hist_prices,
    mode='lines',
    name='Historical Open Price',
    line=dict(color='blue')
))

# Add Predicted Data
fig.add_trace(go.Scatter(
    x=plot_dates,
    y=plot_prices,
    mode='lines+markers',
    name=f'Predicted {PREDICT_DAYS}-Day Trend',
    line=dict(color='orange', dash='dash')
))

fig.update_layout(
    title=f'{TICKER} - {HISTORICAL_DAYS}-Day History & {PREDICT_DAYS}-Day Prediction',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()